# 04 — Tratamento e engenharia de atributos

Vamos implementar as decisões da EDA e criar `comprometimento_renda`, usando funções que também poderão ser executadas na API. O pipeline recebe atributos brutos e aplica sempre a mesma sequência de regras.

**Entrada:** `data/processed/treino.csv`. O teste final permanece reservado. Para demonstrar o isolamento das estatísticas, dividimos o treino em ajuste e validação internos. Ainda não treinamos classificadores nem selecionamos um modelo.

**Ordem do pipeline:** regras fixas de qualidade → imputação dos componentes → nova variável → codificação e eventual padronização. Execute com o kernel da `.venv`, após os notebooks 02 e 03.


## 1. Ler o treino e importar as funções

O código em `src/` pode ser reutilizado nos notebooks, no treinamento e na API. O notebook explica as decisões e apresenta os resultados; as regras ficam nos módulos Python para evitar implementações diferentes do mesmo cálculo.


In [1]:
from pathlib import Path
import hashlib
import json
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display

relativo = Path("data/processed/treino.csv")
raiz = None
for pasta in (Path.cwd(), *Path.cwd().parents):
    if (pasta / relativo).is_file():
        raiz = pasta
        break
if raiz is None:
    raise FileNotFoundError("Execute o notebook 02 para gerar os conjuntos.")
if str(raiz) not in sys.path:
    sys.path.insert(0, str(raiz))

from src.features import adicionar_comprometimento_renda
from src.preprocessing import (
    COLUNAS_CATEGORICAS, COLUNAS_MEDIANA,
    criar_preprocessador, tratar_inconsistencias,
)

caminho_treino = raiz / relativo
metadados = json.loads((raiz / "data/processed/separacao.json").read_text())
hash_treino = hashlib.sha256(caminho_treino.read_bytes()).hexdigest()
assert hash_treino == metadados["sha256_treino"]
treino = pd.read_csv(caminho_treino)
X = treino.drop(columns="loan_status")
y = treino["loan_status"]
print(f"Treino disponível: {len(treino):,} registros; alvo separado das entradas.")


Treino disponível: 25,932 registros; alvo separado das entradas.


## 2. Regras fixas para inconsistências e outliers

Decisões implementadas em `tratar_inconsistencias()`:

| Campo | Regra | Justificativa |
| --- | --- | --- |
| Idade | Fora de 18 a 100 anos vira ausente | Convenção educacional para população adulta e tratamento dos extremos suspeitos; não é um limite universal |
| Renda, empréstimo e taxa | Valores não positivos viram ausentes | O fluxo deste projeto pressupõe renda, valor emprestado e taxa positivos; revisar a hipótese em outro produto de crédito |
| Tempo de emprego e histórico | Negativos ou maiores que a idade original viram ausentes | Inconsistência lógica; zero continua válido |
| Numéricos | Não numéricos e infinitos viram ausentes | Evitar cálculo inválido e permitir tratamento explícito de ausência |
| Categorias | Remover espaços nas extremidades; texto vazio vira ausente | Evitar categorias acidentais por formatação |
| Extremos financeiros positivos | Manter os valores | IQR sinaliza raridade, sem comprovar erro; renda alta não será cortada automaticamente |

Não corrigimos a idade para um número que não conhecemos nem excluímos as linhas com base no alvo. A imputação representa uma aproximação estatística, não a recuperação do valor verdadeiro. O índice e a quantidade de registros são preservados.

**Coluna existente:** `loan_percent_income` é desconsiderada. Usaremos a razão recalculada exigida pelo roteiro, evitando duas versões quase redundantes e divergentes do mesmo indicador. Essa decisão foi tomada pela análise do treino, sem consultar o teste.


In [2]:
X_regras = tratar_inconsistencias(X)
assert X_regras.index.equals(X.index)
assert len(X_regras) == len(X)
diagnostico = pd.DataFrame({
    "nulos_antes": X[X_regras.columns].isna().sum(),
    "nulos_apos_regras": X_regras.isna().sum(),
})
diagnostico["novos_nulos"] = diagnostico["nulos_apos_regras"] - diagnostico["nulos_antes"]
display(diagnostico)
print("Regras aplicadas à cópia em memória; nenhum registro removido.")


,nulos_antes,nulos_apos_regras,novos_nulos
person_age,0,5,5
person_income,0,0,0
person_emp_length,727,729,2
loan_amnt,0,0,0
loan_int_rate,2488,2488,0
cb_person_cred_hist_length,0,0,0
person_home_ownership,0,0,0
loan_intent,0,0,0
loan_grade,0,0,0
cb_person_default_on_file,0,0,0


Regras aplicadas à cópia em memória; nenhum registro removido.


## 3. Criar uma validação interna

Dos 25.932 registros de treino, reservamos 20% para uma validação de demonstração, com estratificação e semente 42. Essa validação interna é diferente dos 6.484 registros do teste final.

O `fit` aprenderá estatísticas somente nos registros de ajuste. Na próxima etapa, o pré-processador ficará junto do classificador em um único pipeline, para que a validação cruzada refaça esse isolamento em cada partição.


In [3]:
X_ajuste, X_validacao, y_ajuste, y_validacao = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y,
)
assert X_ajuste.index.intersection(X_validacao.index).empty
print(f"Ajuste interno: {len(X_ajuste):,}")
print(f"Validação interna: {len(X_validacao):,}")


Ajuste interno: 20,745
Validação interna: 5,187


## 4. Imputar, criar a variável e preparar as entradas

`criar_preprocessador()` devolve um pipeline ainda não ajustado:

1. **Regras:** converte inconsistências em ausentes, sem estimar parâmetros.
2. **Imputação:** mediana para os campos numéricos, exceto taxa de juros, que usa média; moda para categorias, se houver nulos. O tempo de emprego tem assimetria positiva e a mediana é menos sensível aos extremos; a taxa tem média e mediana próximas e assimetria de apenas 0,20. Para os demais numéricos, a mediana oferece uma regra de contingência robusta, incluindo as idades invalidadas.
3. **Atributo:** calcula a razão somente após preencher os componentes `loan_amnt` e `person_income` com estatísticas do ajuste.
4. **Codificação:** `OneHotEncoder(handle_unknown="ignore")` nas categorias. Nas numéricas, `StandardScaler` quando `escalonar=True`, opção destinada aos algoritmos sensíveis à escala.

`fit_transform()` ajusta e transforma o treino de ajuste. `transform()` aplica os parâmetros já aprendidos à validação. Não vamos ajustar o pré-processador fora do pipeline do classificador durante a comparação de modelos.


In [4]:
preprocessador = criar_preprocessador(escalonar=True)
matriz_ajuste = preprocessador.fit_transform(X_ajuste)
matriz_validacao = preprocessador.transform(X_validacao)
assert np.isfinite(matriz_ajuste).all()
assert np.isfinite(matriz_validacao).all()
assert matriz_ajuste.shape[1] == matriz_validacao.shape[1]
print("Etapas:", list(preprocessador.named_steps))
print("Matriz de ajuste:", matriz_ajuste.shape)
print("Matriz de validação:", matriz_validacao.shape)


Etapas: ['regras', 'imputacao', 'atributos', 'codificacao']
Matriz de ajuste: (20745, 26)
Matriz de validação: (5187, 26)


## 5. Inspecionar as estatísticas aprendidas

Os valores abaixo pertencem apenas ao ajuste interno, não à base inteira. A transformação da validação não deve mudar essas estatísticas.


In [5]:
etapa_imputacao = preprocessador.named_steps["imputacao"]
imputador_medianas = etapa_imputacao.named_transformers_["medianas"]
estatisticas_antes = imputador_medianas.statistics_.copy()
tabela_imputacao = pd.DataFrame({"coluna": COLUNAS_MEDIANA,
                                "estrategia": "mediana", "valor": estatisticas_antes})
media_taxa = etapa_imputacao.named_transformers_["taxa_media"].statistics_[0]
tabela_imputacao = pd.concat([
    tabela_imputacao,
    pd.DataFrame([{"coluna": "loan_int_rate", "estrategia": "media", "valor": media_taxa}]),
], ignore_index=True)
display(tabela_imputacao.round(4))
_ = preprocessador.transform(X_validacao)
np.testing.assert_array_equal(estatisticas_antes, imputador_medianas.statistics_)
print("A validação não alterou os valores de imputação.")


,coluna,estrategia,valor
0,person_age,mediana,26.0000
1,person_income,mediana,55000.0000
2,person_emp_length,mediana,4.0000
3,loan_amnt,mediana,8000.0000
4,cb_person_cred_hist_length,mediana,4.0000
5,loan_int_rate,media,11.0179


A validação não alterou os valores de imputação.


## 6. Conferir a nova variável antes da codificação

No fluxo real, a função recebe os componentes já imputados. Em uso isolado, renda zero, negativa, ausente ou infinita resulta em `NaN`, sem criar infinito ou atribuir artificialmente zero risco. A função preserva o DataFrame recebido.

| Nova coluna | Tipo | Fórmula | Interpretação |
| --- | --- | --- | --- |
| `comprometimento_renda` | Numérico contínuo | `(loan_amnt / person_income) × 100` | Valor total do empréstimo em relação à renda anual; não é parcela mensal sobre salário |

Por exemplo, empréstimo de 2.000 e renda anual de 10.000 produzem 20%. A moeda precisa ser a mesma nos dois componentes e permanece não informada na base.


In [6]:
antes_encoding = preprocessador[:3].transform(X_ajuste)
display(antes_encoding[["loan_amnt", "person_income", "comprometimento_renda"]].head())
assert not antes_encoding.isna().any().any()
assert "loan_percent_income" not in antes_encoding.columns
np.testing.assert_allclose(
    antes_encoding["comprometimento_renda"],
    antes_encoding["loan_amnt"] / antes_encoding["person_income"] * 100,
)

exemplos = pd.DataFrame({"loan_amnt": [2000, 2000, 2000], "person_income": [10000, 0, np.nan]})
display(adicionar_comprometimento_renda(exemplos))


,loan_amnt,person_income,comprometimento_renda
6669,5000.0,30000.0,16.666667
1767,10000.0,96500.0,10.362694
16943,7200.0,50000.0,14.400000
16312,10000.0,62000.0,16.129032
19001,24000.0,52800.0,45.454545


,loan_amnt,person_income,comprometimento_renda
0,2000,10000.0,20.0
1,2000,0.0,NaN
2,2000,NaN,NaN


## 7. Preparação para modelos baseados em árvores

Árvores não exigem padronização numérica. A opção `escalonar=False` mantém as mesmas regras, imputação, variável calculada e codificação das categorias, mas passa os números sem `StandardScaler`. Ainda não há classificador nem comparação de desempenho neste notebook.


In [7]:
preprocessador_arvore = criar_preprocessador(escalonar=False)
matriz_arvore = preprocessador_arvore.fit_transform(X_ajuste)
validacao_arvore = preprocessador_arvore.transform(X_validacao)
assert matriz_arvore.shape == matriz_ajuste.shape
assert np.isfinite(validacao_arvore).all()
print("Formato para árvore:", matriz_arvore.shape)
print("Formato para algoritmo com escala:", matriz_ajuste.shape)


Formato para árvore: (20745, 26)
Formato para algoritmo com escala: (20745, 26)


## 8. Preservação dos dados e resultado desta etapa

As transformações foram demonstradas em memória. Não exportamos uma tabela globalmente imputada para usar na validação cruzada: isso poderia levar estatísticas de uma partição a outra. A comparação dos modelos receberá os dados brutos e aprenderá as transformações dentro de cada partição.

As regras e o cálculo são funções importáveis, o que permite salvar um pipeline completo e carregá-lo na API sem reimplementar a transformação. Os testes em `tests/test_preprocessing.py` verificam divisão por zero, preservação da entrada, categorias novas, isolamento das estatísticas e serialização com `joblib`.

**Próximo passo:** comparar pelo menos duas famílias de modelos com quatro valores de complexidade por família, tratamento de desbalanceamento no treino e validação interna para escolher as configurações.


In [8]:
assert hashlib.sha256(caminho_treino.read_bytes()).hexdigest() == hash_treino
assert hashlib.sha256((raiz / "data/raw/credit_risk_dataset.csv").read_bytes()).hexdigest() == metadados["sha256_original"]
print("Treino e original preservados; CSV de teste não utilizado nesta etapa.")


Treino e original preservados; CSV de teste não utilizado nesta etapa.
